# Exploring the AI Entity Context endpoint

`GET /api/v1/{entityType}/name/{fqn}/context` returns the full
**Context Profile** of an asset in one call: it walks the knowledge graph — schema, column-level
lineage, data profile, data quality, glossary, tags, and attached Context Center articles — and
renders it as Markdown (for an LLM) or JSON (for code). The response is **RBAC-filtered for the
caller**, so PII column profiles are hidden unless you own the asset / are admin — section 6
proves that with two tokens against the same URL.

Over MCP the same document arrives as the `context` section of
`get_entity_details(entityType, fqn, include=["context"])` — see
[`persona_demo.ipynb`](./persona_demo.ipynb) for the agent-facing side.

Validated live against the [Jaffle Shop demo database](../resources/demo-database/) ingested into
OpenMetadata and configured by [`setup_demo.py`](./setup_demo.py).

## Prerequisites
```bash
# 1. the demo database, ingested into OpenMetadata as a service named "jaffle shop"
make demo-database && make demo-dbt      # from the repo root

# 2. the demo configured (PII tags, profiles, data-quality results, personas, users)
export AI_SDK_HOST="http://localhost:8585"
export AI_SDK_TOKEN="<admin-jwt>"
python setup_demo.py

# 3. this notebook
pip install requests
```

`setup_demo.py` prints a `DEMO_COMPLIANCE_TOKEN` and a `DEMO_ENGINEER_TOKEN` — section 6 needs
both. Put them in a `.env` next to this notebook (see `.env.example`).

In [7]:
import os
from pathlib import Path

# Secrets live in a gitignored .env next to this notebook — never commit them.
# Copy .env.example to .env and fill in your values (one KEY=value per line).
env_path = Path(".env")
if env_path.exists():
    for line in env_path.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        os.environ.setdefault(key.strip(), value.strip().strip('"').strip("'"))
    print(f"Loaded secrets from {env_path.resolve()}")
else:
    print(".env not found — copy .env.example to .env and fill in your values.")

Loaded secrets from /Users/pmbrull/conductor/workspaces/ai-sdk/athens/cookbook/persona-aware-context/.env


In [8]:
import os
from urllib.parse import quote

import requests

HOST = os.environ.get("AI_SDK_HOST", "http://localhost:8585").rstrip("/")
TOKEN = os.environ["AI_SDK_TOKEN"]  # read from env — never hard-code a token here

# The demo asset. Every section below reads this one table.
TABLE = os.environ.get(
    "DEMO_TABLE_FQN", "jaffle shop.jaffle_shop.marts_core.dim_customers"
)


def get_context(fqn, entity_type="tables", fmt="markdown", query=None, token=None):
    """AI Entity Context for one asset.

    fmt="markdown" -> str (OKF document);  fmt="json" -> dict (AIContext).
    query          -> optional natural-language question that biases the excerpt
                      of *truncated* attached knowledge toward the relevant passage.
    token          -> call as somebody else; defaults to AI_SDK_TOKEN.
    """
    url = f"{HOST}/api/v1/{entity_type}/name/{quote(fqn, safe='')}/context"
    params = {"format": fmt}
    if query:
        params["query"] = query
    headers = {"Authorization": f"Bearer {token or TOKEN}"}
    resp = requests.get(url, headers=headers, params=params, timeout=30)
    resp.raise_for_status()
    return resp.json() if fmt == "json" else resp.text


print(f"{HOST} -> {TABLE}")

http://localhost:8585 -> jaffle shop.jaffle_shop.marts_core.dim_customers


## 1. The rendered Context Profile (Markdown)

This is what you drop straight into an LLM's context window.

In [9]:
from IPython.display import Markdown, display

display(Markdown(get_context(TABLE)))

---
type: "table"
title: "jaffle shop.jaffle_shop.marts_core.dim_customers"
description: "Customer dimension table with lifetime metrics and segmentation."
fullyQualifiedName: "jaffle shop.jaffle_shop.marts_core.dim_customers"
tags: ["PII.Sensitive"]
timestamp: "2026-09-10T09:13:41.630Z"
---

Customer dimension table with lifetime metrics and segmentation.
Use for customer analysis, cohort analysis, and targeting.

# Schema

| Column | Type | Constraint | Description |
|--------|------|------------|-------------|
| customer_id | integer | NULL | Unique customer identifier |
| first_name | text | NULL |  |
| last_name | text | NULL |  |
| full_name | text | NULL |  |
| email | text | NULL | Customer email (PII - may be null for data quality issues) |
| phone_number | character varying(20) | NULL |  |
| city | text | NULL |  |
| state | text | NULL |  |
| postal_code | text | NULL |  |
| country | text | NULL |  |
| customer_created_at | timestamp without time zone | NULL |  |
| total_orders | bigint | NULL |  |
| completed_orders | bigint | NULL |  |
| cancelled_orders | bigint | NULL |  |
| lifetime_value | numeric | NULL | Total revenue from this customer |
| avg_order_value | numeric | NULL |  |
| first_order_date | date | NULL |  |
| last_order_date | date | NULL |  |
| total_items_purchased | numeric | NULL |  |
| orders_with_coupon | bigint | NULL |  |
| days_as_customer | integer | NULL |  |
| days_since_last_order | integer | NULL |  |
| value_segment | text | NULL | Customer value tier (High/Medium/Low/No Purchases) |
| customer_type | text | NULL | Customer loyalty tier (Loyal/Repeat/New/Prospect) |
| total_support_tickets | bigint | NULL |  |
| avg_support_satisfaction | numeric | NULL |  |

# Data Model

**Type:** `DBT` · **Path:** `models/marts/core/dim_customers.sql` · **Project:** `jaffle_shop`

```sql
-- Customer dimension with lifetime metrics

with customers as (
    select * from "jaffle_shop"."staging"."stg_jaffle_shop__customers"
),

orders as (
    select * from "jaffle_shop"."intermediate"."int_orders__enriched"
),

customer_orders as (
    select
        customer_id,
        count(distinct order_id) as total_orders,
        sum(case when order_status = 'completed' then 1 else 0 end) as completed_orders,
        sum(case when order_status = 'cancelled' then 1 else 0 end) as cancelled_orders,
        sum(net_order_value) as lifetime_value,
        avg(net_order_value) as avg_order_value,
        min(order_date) as first_order_date,
        max(order_date) as last_order_date,
        sum(total_units) as total_items_purchased,
        sum(case when used_coupon then 1 else 0 end) as orders_with_coupon
    from orders
    where customer_id is not null
    group by customer_id
),

tickets as (
    select
        customer_id,
        count(*) as total_tickets,
        avg(satisfaction_score) as avg_satisfaction
    from "jaffle_shop"."staging"."stg_support__tickets"
    where customer_id is not null
    group by customer_id
)

select
    c.customer_id,
    c.first_name,
    c.last_name,
    c.first_name || ' ' || c.last_name as full_name,
    c.email,
    c.phone_number,
    c.city,
    c.state,
    c.postal_code,
    c.country,
    c.created_at as customer_created_at,

    -- Order metrics
    coalesce(co.total_orders, 0) as total_orders,
    coalesce(co.completed_orders, 0) as completed_orders,
    coalesce(co.cancelled_orders, 0) as cancelled_orders,
    coalesce(co.lifetime_value, 0) as lifetime_value,
    co.avg_order_value,
    co.first_order_date,
    co.last_order_date,
    coalesce(co.total_items_purchased, 0) as total_items_purchased,
    coalesce(co.orders_with_coupon, 0) as orders_with_coupon,

    -- Derived metrics
    case
        when co.first_order_date is not null
        then 
  (co.last_order_date - co.first_order_date)

        else null
    end as days_as_customer,
    
  (current_date - coalesce(co.last_order_date, CAST(c.created_at AS date)))
 as days_since_last_order,

    -- Customer segments
    case
        when coalesce(co.lifetime_value, 0) >= 200 then 'High Value'
        when coalesce(co.lifetime_value, 0) >= 100 then 'Medium Value'
        when coalesce(co.lifetime_value, 0) > 0 then 'Low Value'
        else 'No Purchases'
    end as value_segment,

    case
        when coalesce(co.total_orders, 0) >= 5 then 'Loyal'
        when coalesce(co.total_orders, 0) >= 2 then 'Repeat'
        when coalesce(co.total_orders, 0) = 1 then 'New'
        else 'Prospect'
    end as customer_type,

    -- Support metrics
    coalesce(t.total_tickets, 0) as total_support_tickets,
    t.avg_satisfaction as avg_support_satisfaction

from customers c
left join customer_orders co on c.customer_id = co.customer_id
left join tickets t on c.customer_id = t.customer_id
```

# Business Definitions

### Data Retention
`DataGovernance.DataRetention`

Policy governing how long customer data may be stored before deletion.

### Personally Identifiable Information
`DataGovernance.PersonallyIdentifiableInformation`

Data that can identify a specific individual — e.g. email, phone number, date of birth, or the last four digits of a national ID.

# Knowledge Articles

### Customer 360 Data Model
`customer-360-data-model`

Governance, operations, and quality notes for the certified customer master.

Sections: Customer 360 Data Model · Sensitive fields, access control, and retention · Refresh cadence, freshness SLA, and upstream dependencies · Known data-quality issues and the dbt test suite

_Excerpt — fetch the full content with get_knowledge_content(entityType=`page`, fqn=`customer-360-data-model`)._

### compliance-operating-rule
`compliance-operating-rule`

When explaining any dataset, always:
- Lead with its data owner, domain, and governance status.
- Cite the governing policy (e.g. the Data Retention policy) and the relevant glossary term.
- Recommend the certified / approved dataset rather than a raw table.

_⚠ Stale — assetUpdatedAfterKnowledge, dataQualityFailing. Weigh this against the asset's current state before relying on it for decisions._

### engineering-operating-rule
`engineering-operating-rule`

When explaining any dataset, always:
- Give the dbt model path (e.g. `models/marts/core/dim_customers.sql`).
- State the freshness SLA and the frequent join keys.
- Include a short, runnable `SELECT` the reader can paste.

_⚠ Stale — assetUpdatedAfterKnowledge, dataQualityFailing. Weigh this against the asset's current state before relying on it for decisions._

# Lineage

**Upstream:**
- `jaffle shop.jaffle_shop.intermediate.int_orders__enriched`
  - `order_status → cancelled_orders`
  - `order_date → first_order_date`
  - `order_id → total_orders`
  - `total_units → total_items_purchased`
  - `net_order_value → avg_order_value`
  - `order_date → last_order_date`
  - `order_date → days_as_customer`
  - `order_id → customer_type`
  - `order_date → days_since_last_order`
  - `net_order_value → value_segment`
  - `used_coupon → orders_with_coupon`
  - `net_order_value → lifetime_value`
  - `order_status → completed_orders`
- `jaffle shop.jaffle_shop.staging.stg_support__tickets`
  - `satisfaction_score → avg_support_satisfaction`
- `jaffle shop.jaffle_shop.staging.stg_jaffle_shop__customers`
  - `customer_id → customer_id`
  - `first_name → first_name`
  - `created_at → customer_created_at`
  - `phone_number → phone_number`
  - `postal_code → postal_code`
  - `last_name → last_name`
  - `first_name → full_name`
  - `created_at → days_since_last_order`
  - `state → state`
  - `email → email`
  - `country → country`
  - `last_name → full_name`
  - `city → city`


# Data Profile

**Row count:** 25

| Column | Null % | Distinct | Min | Max |
|--------|--------|----------|-----|-----|
| customer_id | 0% | 25 | 1.0 | 25.0 |
| first_name | 4% | 23 |  |  |
| last_name | 0% | 24 |  |  |
| full_name | 4% | 23 |  |  |
| email | 8% | 22 |  |  |
| phone_number | 4% | 23 |  |  |
| city | 4% | 22 |  |  |
| state | 4% | 19 |  |  |
| postal_code | 4% | 22 |  |  |
| country | 0% | 1 |  |  |
| customer_created_at | 0% | 24 |  |  |
| total_orders | 0% | 8 | 1.0 | 9.0 |
| completed_orders | 0% | 8 | 1.0 | 9.0 |
| cancelled_orders | 0% | 2 | 0.0 | 1.0 |
| lifetime_value | 0% | 25 | 27.98 | 449.19 |
| avg_order_value | 0% | 25 | 23.30666666666667 | 84.0375 |
| first_order_date | 0% | 24 |  |  |
| last_order_date | 0% | 22 |  |  |
| total_items_purchased | 0% | 13 | 4.0 | 27.0 |
| orders_with_coupon | 0% | 5 | 0.0 | 4.0 |
| days_as_customer | 0% | 16 | 0.0 | 98.0 |
| days_since_last_order | 0% | 22 | 866.0 | 915.0 |
| value_segment | 0% | 3 |  |  |
| customer_type | 0% | 3 |  |  |
| total_support_tickets | 0% | 3 | 0.0 | 2.0 |
| avg_support_satisfaction | 64% | 4 | 3.0 | 5.0 |

# Data Quality

Tests — passed: 4, failed: 1, aborted: 0

> 1 data-quality test(s) are currently failing on this asset — qualify any answer accordingly.


## 2. The same profile as JSON

Every field of the `AIContext` object, for programmatic use.

In [10]:
ctx = get_context(TABLE, fmt="json")

print("Top-level fields:", ", ".join(ctx))
print()
print("description   :", (ctx.get("description") or "").strip())
print("upstream      :", ctx.get("upstream"))
print("downstream    :", ctx.get("downstream"))
print("tags          :", [t.get("tagFQN", t) if isinstance(t, dict) else t for t in ctx.get("tags") or []])
print("glossaryTerms :", [g.get("name") for g in ctx.get("glossaryTerms") or []])
print("articles      :", [a.get("name") for a in ctx.get("articles") or []])

Top-level fields: id, name, fullyQualifiedName, entityType, service, serviceType, description, tags, glossaryTerms, metrics, articles, upstream, downstream, upstreamEdges, downstreamEdges, assetContext, observability, generatedAt

description   : Customer dimension table with lifetime metrics and segmentation.
Use for customer analysis, cohort analysis, and targeting.
upstream      : ['jaffle shop.jaffle_shop.intermediate.int_orders__enriched', 'jaffle shop.jaffle_shop.staging.stg_support__tickets', 'jaffle shop.jaffle_shop.staging.stg_jaffle_shop__customers']
downstream    : []
tags          : ['PII.Sensitive']
glossaryTerms : ['DataRetention', 'PersonallyIdentifiableInformation']
articles      : ['customer-360-data-model', 'compliance-operating-rule', 'engineering-operating-rule']


## 3. Traverse the graph: column-level lineage

The profile names its neighbours — with column mappings — so an agent can hop to the next node's
`/context` and keep walking.

In [11]:
for edge in ctx.get("upstreamEdges") or []:
    print("UP  ", edge["fullyQualifiedName"])
    for mapping in (edge.get("columns") or [])[:3]:
        sources = ", ".join(c.rsplit(".", 1)[-1] for c in mapping["fromColumns"])
        print("       ", sources, "->", mapping["toColumn"].rsplit(".", 1)[-1])

UP   jaffle shop.jaffle_shop.intermediate.int_orders__enriched
        order_status -> cancelled_orders
        order_date -> first_order_date
        order_id -> total_orders
UP   jaffle shop.jaffle_shop.staging.stg_support__tickets
        satisfaction_score -> avg_support_satisfaction
UP   jaffle shop.jaffle_shop.staging.stg_jaffle_shop__customers
        customer_id -> customer_id
        first_name -> first_name
        created_at -> customer_created_at


## 4. Schema, data profile, and data quality

`assetContext.table` carries the structural view; `observability` carries the profile (RBAC-masked)
and the data-quality standing.

In [12]:
table = ctx["assetContext"]["table"]
print("columns      :", len(table["columns"]))
print("primaryKey   :", table.get("primaryKey"))
print("frequentJoins:", table.get("frequentJoins"))
print()

observability = ctx.get("observability") or {}
print("rowCount     :", observability.get("rowCount"))
print("dataQuality  :", observability.get("dataQuality"))
print()
for profile in (observability.get("columnProfiles") or [])[:8]:
    null_pct = profile.get("nullProportion") or 0.0
    print(f"  {profile['name']:22} null={null_pct:.2f}  distinct={profile.get('distinctCount')}")

columns      : 26
primaryKey   : []
frequentJoins: []

rowCount     : 25.0
dataQuality  : {'total': 5, 'passed': 4, 'failed': 1, 'aborted': 0}

  customer_id            null=0.00  distinct=25.0
  first_name             null=0.04  distinct=23.0
  last_name              null=0.00  distinct=24.0
  full_name              null=0.04  distinct=23.0
  email                  null=0.08  distinct=22.0
  phone_number           null=0.04  distinct=23.0
  city                   null=0.04  distinct=22.0
  state                  null=0.04  distinct=19.0


### Sample rows

`assetContext.table.sampleData` carries actual rows — but **only in the JSON payload**. The
markdown render has no Sample Data section, so if you only ever look at `format=markdown`
you will conclude the rows aren't there.

In [13]:
sample = table.get("sampleData") or {}
columns = sample.get("columns") or []
rows = sample.get("rows") or []
print(f"{len(rows)} sample row(s), {len(columns)} columns")
print()

show = [columns.index(name) for name in ("customer_id", "full_name", "email", "value_segment")]
print(" | ".join(f"{columns[i]:22}" for i in show))
for row in rows[:5]:
    print(" | ".join(f"{str(row[i]):22}" for i in show))

10 sample row(s), 26 columns

customer_id            | full_name              | email                  | value_segment         
1                      | Michael Perez          | mperez@example.com     | High Value            
2                      | Shawn Myers            | smyers@example.com     | Medium Value          
3                      | Kathleen Johnson       | kjohnson@example.com   | High Value            
4                      | Jimmy Ramirez          | jramirez@example.com   | Medium Value          
5                      | Sara Chen              | schen@example.com      | Medium Value          


## 5. Question-aware context — the `query` param

`query` does **not** filter or change *which* asset you get. When an attached knowledge item (a
glossary definition or a Context Center article) is too long to inline in full, the excerpt shown is
picked by **semantic similarity to `query`** — the most relevant chunk of that item — instead of the
positional lead. It gives the agent a decision-grade snippet of a long doc.

It only *visibly* changes the output when **all** of these hold (server constants):

- the item is **truncated** — body `> ~1500 chars` (`MAX_ITEM_CHARS`), so `contentTruncated=True`;
- it spans **multiple chunks** — the body is chunked every **~380 words** (`MAX_WORDS_PER_CHUNK`), so
  each *topic* needs its own ~400-word section to land in a different chunk;
- **vector embeddings are enabled** on the instance (otherwise it falls back to a structural preview).

`setup_demo.py` attaches the `customer-360-data-model` article to `dim_customers` for exactly this
reason: three ~400-word sections — access control and retention, refresh cadence and freshness, data
quality — so the **same asset** returns a **different passage** per question.

In [8]:
ARTICLE = "customer-360-data-model"
QUESTIONS = [
    None,
    "how long do we keep customer email and phone before deletion?",
    "when does the table refresh and what is the freshness SLA?",
    "why do so many rows have a null email and can I trust value_segment?",
]


def article_excerpt(fqn, article_name, query=None):
    """Return the (possibly query-biased) excerpt of one attached article."""
    context = get_context(fqn, fmt="json", query=query)
    for article in context.get("articles") or []:
        if article.get("name") == article_name:
            content = article.get("content") or ""
            # a query excerpt carries a "name: ...; | <body>" chunk header — keep the body
            body = content.split(" | ", 1)[-1].strip()
            return article.get("contentTruncated"), body
    return None, ""


for question in QUESTIONS:
    truncated, body = article_excerpt(TABLE, ARTICLE, query=question)
    label = "NO QUERY (positional lead)" if question is None else f"query = {question!r}"
    print("=" * 95)
    print(f"{label}   (truncated={truncated})")
    print("-" * 95)
    print(body[:320].replace("&#96;", "`"), "...\n")

NO QUERY (positional lead)   (truncated=True)
-----------------------------------------------------------------------------------------------
Governance, operations, and quality notes for the certified customer master.

Sections: Customer 360 Data Model · Sensitive fields, access control, and retention · Refresh cadence, freshness SLA, and upstream dependencies · Known data-quality issues and the dbt test suite ...



query = 'how long do we keep customer email and phone before deletion?'   (truncated=True)
-----------------------------------------------------------------------------------------------
title: Customer 360 Data Model; description: # Customer 360 Data Model Governance, operations, and quality notes for the certified customer master. ## Sensitive fields, access control, and retention `dim_customers` is the certified customer master for the Jaffle Shop warehouse and it carries directly identifyin ...



query = 'when does the table refresh and what is the freshness SLA?'   (truncated=True)
-----------------------------------------------------------------------------------------------
dbt model materialised as a table in the `marts_core` schema. The source of record is `raw_jaffle_shop.customers`, which is loaded by the ingestion job that lands the operational Postgres extract. From there the model passes through `staging.stg_jaffle_shop__customers`, where types are cast, whi ...

query = 'why do so many rows have a null email and can I trust value_segment?'   (truncated=True)
-----------------------------------------------------------------------------------------------
to distrust the whole table. The most visible issue is null contact detail. Roughly eight percent of rows have a null `email` and about four percent have a null `phone_number`, `full_name`, `city`, `state` or `postal_code`. These are genuine gaps in the operational source ...



## 6. The same URL, two callers — RBAC is server-side

Nothing below changes except the bearer token. `setup_demo.py` made **David Kim** an owner of
`dim_customers` and tagged its sensitive columns `PII.Sensitive`; **Sara Johnson** is a non-owner
with the same `DataConsumer` role.

OpenMetadata filters two things for anyone who is not an admin, a bot, or an owner, and it does so
**before serialising the response** — so an assistant answering on Sara's behalf never receives the
values over the wire and cannot leak them, however the prompt is worded:

- **Column profiles** — dropped per column, only for the ones tagged `PII.Sensitive`.
- **Sample rows** — masked **wholesale**. One sensitive column is enough to redact every value in
  every row, because a row is only as shareable as its most sensitive field.

In [9]:
CALLERS = {
    "David Kim (owner)": os.environ.get("DEMO_COMPLIANCE_TOKEN"),
    "Sara Johnson (non-owner)": os.environ.get("DEMO_ENGINEER_TOKEN"),
}

if not all(CALLERS.values()):
    print("Set DEMO_COMPLIANCE_TOKEN and DEMO_ENGINEER_TOKEN (setup_demo.py prints both).")
else:
    profiled = {}
    masked = {}
    for label, token in CALLERS.items():
        payload = get_context(TABLE, fmt="json", token=token)
        observability = payload.get("observability") or {}
        profiled[label] = {p["name"] for p in observability.get("columnProfiles") or []}
        caller_sample = (payload["assetContext"]["table"].get("sampleData") or {}).get("columns") or []
        masked[label] = sum("[MASKED]" in name for name in caller_sample)
        print(
            f"{label:26} -> {len(profiled[label]):2} column profiles"
            f" · {masked[label]:2}/{len(caller_sample)} sample columns masked"
        )

    owner, non_owner = profiled.values()
    print()
    print("Profiles visible only to the owner:", ", ".join(sorted(owner - non_owner)) or "(none)")

David Kim (owner)          -> 26 column profiles ·  0/26 sample columns masked
Sara Johnson (non-owner)   -> 20 column profiles · 26/26 sample columns masked

Profiles visible only to the owner: email, first_name, full_name, last_name, phone_number, postal_code
